# 002 — Well EEI Comparison

Demonstrates the Extended Elastic Impedance (EEI) chi-rotation contract shared between
the well domain and the seismic domain.

**Convention (Ball et al. 2014, Table 1):**
- All chi angles in degrees, via `arctan2(W_Is, W_Ip)`
- Background k = Vs²/Vp² = 0.25  (Vp/Vs ≈ 2, Poisson's ratio ≈ 1/3)
- χ = 0°  → rel_Ip  (P-impedance proxy)
- χ = 90° → rel_Is  (S-impedance / shear proxy)
- χ = –45° → rel_Vp/Vs proxy (fluid indicator)

**Well:** 6607/3-1 S (North Sea) — P-sonic, S-sonic, density

**Seismic comparison interface:**  
Seismic RAI (Relative Acoustic Impedance) extracted at iline=4272, xline=10486 ≈ rel_EEI at χ=0°.
When the seismic cube is available, load `Attributes/Attributes_Horizons.seisnc` and
call `sample_cube_near_well()` (see notebook 001_seismic_masking).

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import lasio
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from qsi.seismic.rrp import (
    decompose,
    compute_rel_eei,
    chi_for_property,
)

In [ ]:
# ── Load well ─────────────────────────────────────────────────────────────────
LAS_PATH = pathlib.Path("../relative-rock-physics/66073-1 S.las")
las = lasio.read(str(LAS_PATH), ignore_header_errors=True)

depth = np.asarray(las.index, dtype=float)  # metres MD

def _dt_to_kms(arr):
    """Convert slowness (µs/ft) to velocity (km/s)."""
    a = np.where(arr > 0, arr, np.nan).astype(float)
    return 304.8 / a

# ML-filled curves: most complete, gap-free
vp  = _dt_to_kms(las["AC_ML_FILLED"])
vs  = _dt_to_kms(las["ACS_ML_FILLED"])
rho = np.where(las["DEN_ML_FILLED"] > 0, las["DEN_ML_FILLED"], np.nan).astype(float)

ip  = vp  * rho   # P-impedance  [km/s · g/cc = GPa / (km/s)  (same ratio as acoustic impedance])
is_ = vs  * rho   # S-impedance

valid = np.isfinite(ip) & np.isfinite(is_)
print(f"Well: {las.well['WELL'].value}")
print(f"Depth range: {depth[valid].min():.1f} – {depth[valid].max():.1f} m  ({valid.sum()} samples)")
print(f"Vp  {np.nanmedian(vp):.3f} km/s  |  Vs {np.nanmedian(vs):.3f} km/s  |  ρ {np.nanmedian(rho):.3f} g/cc")
print(f"Ip  {np.nanmedian(ip):.3f}  |  Is {np.nanmedian(is_):.3f}")

In [ ]:
# ── HP decomposition ──────────────────────────────────────────────────────────
# Le controls the HP filter window.  200 samples @ 0.1524 m/sample ≈ 30 m,
# which approximates the seismic bandwidth pass (tuning thickness order).
Le = 200

ip_v  = ip[valid]
is_v  = is_[valid]
d_v   = depth[valid]

rel_ip,  trend_ip,  _  = decompose(ip_v,  Le=Le)
rel_is,  trend_is,  _  = decompose(is_v,  Le=Le)

print("rel_Ip  range:", rel_ip.min().round(4), "–", rel_ip.max().round(4))
print("rel_Is  range:", rel_is.min().round(4), "–", rel_is.max().round(4))

In [ ]:
# ── Chi angles for common rock-physics properties ────────────────────────────
CHI_PROPS = ["Ip", "Is", "VpVs", "MuRho", "LamRho"]
chi_angles = {p: chi_for_property(p) for p in CHI_PROPS}
print("Ball et al. (2014) Table 1 — k = 0.25")
for p, chi in chi_angles.items():
    print(f"  {p:<10s}  χ = {chi:+.1f}°")

In [ ]:
# ── Compute rel_EEI at each property angle ───────────────────────────────────
eei_logs = {p: compute_rel_eei(rel_ip, rel_is, chi) for p, chi in chi_angles.items()}

In [ ]:
# ── Plot: rel_EEI log suite ───────────────────────────────────────────────────
N = len(CHI_PROPS)
fig, axes = plt.subplots(1, N, figsize=(3 * N, 10), sharey=True)
fig.suptitle("Well 6607/3-1 S — rel_EEI at key χ angles  (Le=200)", fontsize=11)

for ax, (prop, log) in zip(axes, eei_logs.items()):
    chi = chi_angles[prop]
    ax.plot(log, d_v, lw=0.6, color="steelblue")
    ax.axvline(0, color="k", lw=0.5, ls="--")
    ax.set_title(f"{prop}\nχ = {chi:+.0f}°", fontsize=9)
    ax.set_xlabel("rel_EEI")
    ax.invert_yaxis()
    ax.grid(True, lw=0.3, alpha=0.5)

axes[0].set_ylabel("Depth (m MD)")
plt.tight_layout()
plt.show()

In [ ]:
# ── Seismic comparison interface ──────────────────────────────────────────────
# When the seismic cube (Attributes/Attributes_Horizons.seisnc) is available:
#
#   import xarray as xr
#   cube = xr.open_dataset("Attributes/Attributes_Horizons.seisnc")
#
#   WELL_ILINE, WELL_XLINE = 4272, 10486
#   rai_trace = cube["RelativeAcousticImpedance"].sel(
#       iline=WELL_ILINE, xline=WELL_XLINE, method="nearest"
#   ).values
#   twt_samples = cube.samples.values
#
#   # rel_EEI at χ=0° is the well-domain equivalent of seismic RAI:
#   assert np.allclose(eei_logs["Ip"], compute_rel_eei(rel_ip, rel_is, 0.0))
#
# The seismic trace is in TWT (ms); the well log is in depth (m MD).
# A depth-time conversion (checkshot or velocity model) is needed before
# a quantitative tie.  This cell documents the contract so the wiring is
# in place when the cube and time-depth function become available.

print("Seismic comparison interface — data not present in this repo.")
print("See comment above for the extraction pattern.")
print()
print("Well  iline / xline : 4272 / 10486")
print(f"rel_Ip at χ=0°  → matches seismic RAI after depth-time conversion")
print(f"rel_EEI chi angles: {chi_angles}")